# Домашняя работа №8
Мацюк Константин Владимирович  
НПИбд-02-25  
1032253640

Линейные неоднородные ОДУ с постоянными коэффициентами. Глава 5 пособия.

Ядро: SageMath. Все необходимые функции включены в файл.
Формулы ответов проверены независимо в SymPy; ячейки SageMath в среде подготовки не запускались.

In [ ]:
from sage.all import *
import sympy as sp
t, mu = var('t mu')
assume(t, 'real')
R = PolynomialRing(QQbar, 'mu')
z = R.gen()
B = PolynomialRing(SR, 's')

## Задание 1
Написать функцию, которая по заданному символьному выражению $f$ опознаёт линейную комбинацию выражений вида $t^m e^{\mu t}$ и возвращает дифференциальный оператор $Q$, такой, что $Qf=0$.

Для распознавания синусы и косинусы переводятся в экспоненты средствами SymPy, входящей в SageMath. Далее используются полиномы и символьные выражения SageMath. Показатели экспонент и коэффициенты оператора $P$ предполагаются алгебраическими, как в предыдущей работе; $m$ — неотрицательное целое число.

In [ ]:
def exponential_terms(expr, T):
    # Результат: показатель mu -> многочлен при exp(mu*T).
    expr = sp.expand(expr.rewrite(sp.exp))
    terms = {}
    for term in sp.Add.make_args(expr):
        rate, coefficient = sp.S.Zero, sp.S.One
        for factor in sp.Mul.make_args(term):
            if factor.func == sp.exp:
                arg = sp.expand(factor.args[0])
                a = sp.diff(arg,T)
                b = sp.simplify(arg-a*T)
                if a.has(T) or b.has(T):
                    raise ValueError('Показатель экспоненты должен быть линейным по t.')
                rate += a
                coefficient *= sp.exp(b)
            else:
                coefficient *= factor
        try:
            polynomial = sp.Poly(coefficient,T)
        except sp.PolynomialError:
            raise ValueError('Ожидается сумма многочленов по t, умноженных на экспоненты.')
        rate = sp.simplify(rate)
        terms[rate] = terms.get(rate,sp.S.Zero)+polynomial.as_expr()
    result = {}
    for a,p in terms.items():
        p = sp.Poly(sp.simplify(p),T)
        if not p.is_zero:
            result[a] = p
    return result

In [ ]:
def exp_terms(f):
    terms = exponential_terms(SR(f)._sympy_(),t._sympy_())
    result = {}
    for a,p in terms.items():
        rate = QQbar(SR(a))
        polynomial = B([SR(p.nth(k)) for k in range(p.degree()+1)])
        result[rate] = result.get(rate,B.zero())+polynomial
    return {a:p for a,p in result.items() if p != 0}

def apply_operator(p, f):
    f = SR(f)
    return SR(sum(SR(p[k])*diff(f,t,k) for k in range(p.degree()+1))).full_simplify()

def annihilator(f):
    q = R.one()
    for a,p in exp_terms(f).items():
        q *= (z-a)**(p.degree()+1)
    def Q(g):
        return apply_operator(q,g)
    Q.symbol = q
    return Q

In [ ]:
forces = [exp(t),2*exp(-t),3*exp(-t),exp(t)+2*sin(t)]
for f in forces:
    Q = annihilator(f)
    print('f =',f)
    print('Символ Q:',Q.symbol.factor())
    print('Q(f) =',Q(f))

f = (1+t**2)*exp(2*t)+t*sin(t)
Q = annihilator(f)
print('Дополнительный пример:',Q.symbol.factor())
print('Q(f) =',Q(f))

Ответы для четырёх правых частей:
$$Q_1=D-1,\qquad Q_2=Q_3=D+1,\qquad Q_4=(D-1)(D^2+1).$$
Для дополнительного примера:
$$Q=(D-2)^3(D^2+1)^2.$$
Во всех случаях $Q(f)=0$. Для $f=0$ функция возвращает единичный оператор. Выражения вне указанного класса отклоняются с сообщением об ошибке.

## Задание 2
Написать функцию, которая по заданному оператору $P$ и выражению $f$ из предыдущего номера возвращает общее решение уравнения $Px=f$.

Используется метод из §5.3: строится $Q$, затем базис решений $QP x=0$. Коэффициенты дополнительных к ядру $P$ функций определяются подстановкой в исходное уравнение.

In [ ]:
def operator_symbol(P):
    p = R(SR(P(exp(mu*t))*exp(-mu*t)).expand())
    if p == 0:
        raise ValueError('Оператор P должен быть ненулевым.')
    return p

def real_constant(a):
    a = QQbar(a)
    e = SR(a.radical_expression())
    return SR(AA(a)) if e.has(I) else e

def root_functions(multiplicities, shifts=None, real_form=True):
    shifts = {} if shifts is None else shifts
    basis = []
    for a,m in multiplicities.items():
        start = shifts.get(a,0)
        for j in range(start,start+m):
            if not real_form:
                basis.append(t**j*exp(SR(a.radical_expression())*t))
            elif a.imag() == 0:
                basis.append(t**j*exp(real_constant(a.real())*t))
            elif a.imag() > 0:
                alpha = real_constant(a.real())
                beta = real_constant(a.imag())
                basis.extend([t**j*exp(alpha*t)*cos(beta*t),
                              t**j*exp(alpha*t)*sin(beta*t)])
    return basis

def solution_parts(P, f):
    p = operator_symbol(P)
    q = annihilator(f).symbol
    proots, qroots = dict(p.roots()), dict(q.roots())
    real_form = all(a.imag() == 0 for a in list(p)+list(q))
    homogeneous = root_functions(proots,real_form=real_form)
    additional = root_functions(qroots,shifts=proots,real_form=real_form)
    if not additional:
        return homogeneous, SR(0)
    a = [SR.var('a'+str(i)) for i in range(len(additional))]
    trial = sum(c*g for c,g in zip(a,additional))
    residual = apply_operator(p,trial)-f
    equations = [c == 0 for pol in exp_terms(residual).values()
                 for c in pol.list() if c != 0]
    coefficients = solve(equations,a,solution_dict=True)[0]
    particular = SR(trial.subs(coefficients)).full_simplify()
    return homogeneous, particular

def general_solution(P, f):
    basis, particular = solution_parts(P,SR(f))
    C = [SR.var('C'+str(i+1)) for i in range(len(basis))]
    return particular+sum(c*g for c,g in zip(C,basis))

## Задание 3
При помощи этой функции найти общие решения ЛОДУ:

а) $x^{(3)}+x=e^t$;

б) $x^{(3)}+x=2e^{-t}$;

в) $x^{(3)}+3x''+3x'+x=3e^{-t}$;

г) $x''+x=e^t+2\sin t$.

In [ ]:
operators = [
    lambda x: diff(x,t,3)+x,
    lambda x: diff(x,t,3)+x,
    lambda x: diff(x,t,3)+3*diff(x,t,2)+3*diff(x,t)+x,
    lambda x: diff(x,t,2)+x
]
labels = ['а','б','в','г']
general = [general_solution(P,f) for P,f in zip(operators,forces)]
for label,P,f,x in zip(labels,operators,forces,general):
    print(label+')')
    show(x)
    print('P(x)-f =',SR(P(x)-f).full_simplify())

Ответы (с точностью до переобозначения произвольных констант):

а)
$$x=C_1e^{-t}+e^{t/2}\left(C_2\cos\frac{\sqrt3t}{2}+C_3\sin\frac{\sqrt3t}{2}\right)+\frac12e^t.$$
б)
$$x=C_1e^{-t}+e^{t/2}\left(C_2\cos\frac{\sqrt3t}{2}+C_3\sin\frac{\sqrt3t}{2}\right)+\frac23te^{-t}.$$
в)
$$x=e^{-t}\left(C_1+C_2t+C_3t^2+\frac{t^3}{2}\right).$$
г)
$$x=C_1\cos t+C_2\sin t+\frac12e^t-t\cos t.$$
Подстановка каждого ответа в соответствующее уравнение даёт нулевую невязку.

## Задание 4
Написать функцию, которая по заданному оператору $P$, выражению $f$ из предыдущего номера и начальным данным возвращает решение задачи Коши для уравнения $Px=f$.

Список `values` содержит $x(0),x'(0),\ldots$. Первые $n$ значений определяют решение ОДУ порядка $n$; дополнительные значения проверяются на совместимость.

In [ ]:
def cauchy(P, f, values):
    p = operator_symbol(P)
    n = p.degree()
    if len(values) < n:
        raise ValueError('Недостаточно начальных условий.')
    basis, particular = solution_parts(P,SR(f))
    C = [SR.var('C'+str(i+1)) for i in range(n)]
    x = particular+sum(c*g for c,g in zip(C,basis))
    if n > 0:
        equations = [diff(x,t,k).subs(t=0) == values[k] for k in range(n)]
        constants = solve(equations,C,solution_dict=True)[0]
        x = SR(x.subs(constants)).full_simplify()
    for k in range(n,len(values)):
        actual = SR(diff(x,t,k).subs(t=0)).full_simplify()
        if SR(actual-values[k]).full_simplify() != 0:
            raise ValueError('Несовместимые условия: производная порядка %s '
                             'в нуле должна быть %s, задано %s.' % (k,actual,values[k]))
    return x

## Задание 5
При помощи этой функции найти решения начальных задач:

а) $x^{(3)}+x=e^t$, $x(0)=0$, $x'(0)=1$, $x''(0)=0$;

б) $x^{(3)}+x=2e^{-t}$, $x(0)=0$, $x'(0)=1$, $x''(0)=2$;

в) $x^{(3)}+3x''+3x'+x=3e^{-t}$, $x(0)=-1$, $x'(0)=1$, $x''(0)=0$;

г) $x''+x=e^t+2\sin t$, $x(0)=0$, $x'(0)=0$, $x''(0)=0$.

In [ ]:
initials = [[0,1,0],[0,1,2],[-1,1,0],[0,0,0]]
for label,P,f,u0 in zip(labels,operators,forces,initials):
    print(label+')')
    try:
        x = cauchy(P,f,u0)
        show(x)
        print('P(x)-f =',SR(P(x)-f).full_simplify())
        print('Начальные значения:',
              [SR(diff(x,t,k).subs(t=0)).full_simplify() for k in range(len(u0))])
    except ValueError as error:
        print(error)

Ответы:

а)
$$x=\frac{e^t-e^{-t}}2.$$
б)
$$x=\left(1+\frac23t\right)e^{-t}
+e^{t/2}\left(-\cos\frac{\sqrt3t}{2}+\frac{11\sqrt3}{9}\sin\frac{\sqrt3t}{2}\right).$$
в)
$$x=e^{-t}\left(-1+\frac{t^2}{2}+\frac{t^3}{2}\right).$$
г) **Решения нет: начальные условия несовместимы.** Подставляя $t=0$ в уравнение, получаем
$$x''(0)+x(0)=1,$$
тогда как заданные значения дают $0+0=0$.

Дополнительно рассмотрим вариант с первыми двумя условиями $x(0)=x'(0)=0$; третье условие в этом варианте не задаётся.

In [ ]:
x = cauchy(operators[3],forces[3],[0,0])
show(x)
print('Начальные значения порядков 0, 1, 2:',
      [SR(diff(x,t,k).subs(t=0)).full_simplify() for k in range(3)])

Для варианта с двумя условиями:
$$x=\frac12e^t-\frac12\cos t+\frac12\sin t-t\cos t.$$
У этого решения $x''(0)=1$, поэтому исходному третьему условию оно не удовлетворяет.